# SE3082 Lab 09 - Introduction to CUDA

**Student:** IT24101492

Run this notebook in Google Colab with a GPU runtime. The source cells are kept in sync with the separate `.cu` files in this repository. The notebook has no saved execution output yet; run its cells in Colab to produce real output from your assigned GPU.

The compile cells use `-arch=sm_75` for a Tesla T4. If Colab assigns a different GPU, use its compute capability from Exercise 1 to select the matching `sm_XX` architecture.

## Exercise 1 - Set up and check the Colab CUDA environment

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or the available GPU) and save. Run these checks. If `nvidia-smi` shows a different GPU, use the compute capability query to choose the correct architecture flag.

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

The compute capability 7.5 maps to `sm_75`, 8.0 to `sm_80`, 8.9 to `sm_89`, and 9.0 to `sm_90`. The small smoke test below should print four lines, one from each GPU thread.

In [ ]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello()
{
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main()
{
    hello<<<1, 4>>>();
    cudaError_t error = cudaGetLastError();
    if (error != cudaSuccess) {
        std::fprintf(stderr, "Kernel launch failed: %s\n", cudaGetErrorString(error));
        return 1;
    }
    error = cudaDeviceSynchronize();
    if (error != cudaSuccess) {
        std::fprintf(stderr, "Kernel execution failed: %s\n", cudaGetErrorString(error));
        return 1;
    }
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

## Exercise 2 - Add two integers on the GPU

The CPU allocates device memory, copies `a` and `b` to the GPU, launches one block with one thread, then copies `c` back. Expected output: `Result is 9`.

In [ ]:
%%writefile ex2_add.cu
#include <cstdio>

__global__ void add(const int *a, const int *b, int *c)
{
    *c = *a + *b;
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

int main()
{
    int a = 2, b = 7, c = 0;
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    const size_t bytes = sizeof(int);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, &a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, &b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    add<<<1, 1>>>(d_a, d_b, d_c);
    if (!check_cuda(cudaGetLastError(), "add kernel launch") ||
        !check_cuda(cudaMemcpy(&c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host"))
        return 1;

    std::printf("Result is %d\n", c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return c == 9 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

**Think answers**

1. Device variables are needed because GPU kernels use device memory; ordinary host variables are in CPU memory and cannot be dereferenced by the GPU.
2. `&a` is the host address used as the source of a host-to-device copy. `d_a` is the device address passed to the kernel.
3. The launch has one thread. Changing it to 10 threads would not change the answer if each thread were written to avoid racing on the same output; the current kernel is intentionally designed for one thread.

## Exercise 3 - Vector addition with 512 blocks and one thread each

The program prints each pair and checks the GPU result against the CPU result. The final line should report zero mismatches.

In [ ]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(const int *a, const int *b, int *c)
{
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    add<<<N, 1>>>(d_a, d_b, d_c);
    if (!check_cuda(cudaGetLastError(), "add kernel launch") ||
        !check_cuda(cudaDeviceSynchronize(), "add kernel execution") ||
        !check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host"))
        return 1;

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        std::printf("%d + %d = %d\n", a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i])
            ++errors;
    }
    std::printf("Verification: %d mismatches out of %d\n", errors, N);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

**Predict before running:** `add<<<512, 1>>>` launches 512 blocks and 512 threads total. Threads execute in warps of 32, so a one-thread block leaves the other lanes in its warp inactive; it does not use a warp efficiently.

## Exercise 4 - Vector addition with one block and 512 threads

This version selects the element with `threadIdx.x`. It should verify all 512 results.

In [ ]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(const int *a, const int *b, int *c)
{
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate host arrays.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    addT<<<1, N>>>(d_a, d_b, d_c);
    if (!check_cuda(cudaGetLastError(), "addT kernel launch") ||
        !check_cuda(cudaDeviceSynchronize(), "addT kernel execution") ||
        !check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host"))
        return 1;

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        std::printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i])
            ++errors;
    }
    std::printf("Verification: %d mismatches out of %d\n", errors, N);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

**Find the limit experiment:** the normal program stays at `N = 512`. The companion program below attempts a block of 2,048 threads. On a T4 this exceeds the 1,024 threads-per-block limit, and `cudaGetLastError()` reports an invalid launch configuration. The program outputs the launch error; then the normal 512-thread program can be run again unchanged.

In [ ]:
%%writefile ex4_limit.cu
#include <cstdio>

__global__ void probe()
{
}

int main()
{
    // The device limit is at most 1024 threads in one block.
    probe<<<1, 2048>>>();
    const cudaError_t error = cudaGetLastError();
    if (error == cudaSuccess) {
        std::printf("This GPU accepted 2048 threads in one block.\n");
        return 0;
    }
    std::printf("Kernel launch failed: %s\n", cudaGetErrorString(error));
    std::printf("The single-block thread limit is smaller than 2048.\n");
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_limit.cu -o ex4_limit && ./ex4_limit

## Exercise 5 - Multiply two 10,000,000-element vectors

The kernel computes the global index from block and thread coordinates and guards the final partially filled block. It prints the last 1,000 results and verifies all elements. The expected launch is 19,532 blocks × 512 threads = 10,000,384 total threads; 384 threads are outside the array and must return without accessing it.

In [ ]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>
#include <cstddef>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(const int *a, const int *b, int *c, int n)
{
    const int i = static_cast<int>(blockIdx.x * blockDim.x + threadIdx.x);
    if (i < n)
        c[i] = a[i] * b[i];
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, int count)
{
    for (int i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t bytes = static_cast<size_t>(N) * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host vectors.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, N);
    random_ints(b, N);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    const int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    std::printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    if (!check_cuda(cudaGetLastError(), "vecMul kernel launch") ||
        !check_cuda(cudaDeviceSynchronize(), "vecMul kernel execution") ||
        !check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host"))
        return 1;

    for (int i = N - 1000; i < N; ++i)
        std::printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %d\n", errors, N);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

## Exercise 6 - Element-wise product of two 10,000 × 10,000 matrices

This is element-wise multiplication, not matrix multiplication. It uses flat row-major arrays and a two-dimensional grid. Each block has 32 × 16 = 512 threads. The expected grid is 313 × 625 blocks. Both row and column coordinates are bounds-checked before computing the flat index. This exercise allocates about 1.2 GB across three host arrays and another 1.2 GB across three GPU arrays, so use a Colab GPU runtime with enough memory.

In [ ]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>
#include <cstddef>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(const int *a, const int *b, int *c, int rows, int cols)
{
    const int col = static_cast<int>(blockIdx.x * blockDim.x + threadIdx.x);
    const int row = static_cast<int>(blockIdx.y * blockDim.y + threadIdx.y);
    if (row < rows && col < cols) {
        const int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

static bool check_cuda(cudaError_t error, const char *operation)
{
    if (error == cudaSuccess)
        return true;
    std::fprintf(stderr, "%s failed: %s\n", operation, cudaGetErrorString(error));
    return false;
}

static void random_ints(int *values, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        values[i] = std::rand() % 100;
}

int main()
{
    const size_t count = static_cast<size_t>(ROWS) * COLS;
    const size_t bytes = count * sizeof(int);
    int *a = static_cast<int *>(std::malloc(bytes));
    int *b = static_cast<int *>(std::malloc(bytes));
    int *c = static_cast<int *>(std::malloc(bytes));
    int *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!a || !b || !c) {
        std::fprintf(stderr, "Could not allocate the host matrices.\n");
        std::free(a); std::free(b); std::free(c);
        return 1;
    }
    random_ints(a, count);
    random_ints(b, count);

    if (!check_cuda(cudaMalloc(&d_a, bytes), "cudaMalloc(d_a)") ||
        !check_cuda(cudaMalloc(&d_b, bytes), "cudaMalloc(d_b)") ||
        !check_cuda(cudaMalloc(&d_c, bytes), "cudaMalloc(d_c)"))
        return 1;
    if (!check_cuda(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice), "copy a to device") ||
        !check_cuda(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice), "copy b to device"))
        return 1;

    const dim3 threadsPerBlock(32, 16);
    const dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                         (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    std::printf("Grid: %u x %u blocks\n", numBlocks.x, numBlocks.y);
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    if (!check_cuda(cudaGetLastError(), "mulElem kernel launch") ||
        !check_cuda(cudaDeviceSynchronize(), "mulElem kernel execution") ||
        !check_cuda(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost), "copy result to host"))
        return 1;

    const int last_row = ROWS - 1;
    for (int col = COLS - 1000; col < COLS; ++col) {
        const size_t idx = static_cast<size_t>(last_row) * COLS + col;
        std::printf("C[%d][%d] = %d x %d = %d\n", last_row, col, a[idx], b[idx], c[idx]);
    }

    int errors = 0;
    for (size_t i = 0; i < count; ++i)
        if (c[i] != a[i] * b[i])
            ++errors;
    std::printf("Verification: %d mismatches out of %zu\n", errors, count);

    std::free(a); std::free(b); std::free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return errors == 0 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

The matrix program prints `C[9999][9000]` through `C[9999][9999]` and reports the number of mismatches after checking all 100,000,000 values. The handout expects `Verification: 0 mismatches out of 100000000`. Save this notebook to Drive; Colab’s working disk and compiled programs are temporary.